In [2]:
import numpy as np
from scipy.io import mmread
from gmres1 import gmres
from gmres2 import gmres_givens
from gmres3 import gmres_givens_R

In [8]:
# Example matrix
A= np.array([[4., 1.],[2., 3.]])
b= np.array([1., 2.])
x0= np.zeros(2)

#Q1 test
x_approx, res= gmres(A, b, x0, m=2)
print("Approximate solution:", x_approx)
print("Residual norm:", res)

#Q2 test
x_approx, res= gmres_givens(A, b, x0, m=2)
print("Approximate solution:", x_approx)
print("Residual norm:", res)

#Q3 test
x_approx, res= gmres_givens_R(A, b, x0, m=2, tol=1e-15)
print("Approximate solution:", x_approx)
print("Residual norm:", res)

Approximate solution: [0.1 0.6]
Residual norm: 9.485749680535094e-16
Approximate solution: [0.1 0.6]
Residual norm: 3.66205343881779e-16
Approximate solution: [0.1 0.6]
Residual norm: [np.float64(0.4000000000000001), np.float64(3.66205343881779e-16)]


# Q4
## Experimental Setup

In all numerical experiments, we prescribed an exact solution  

$$
x_{\text{true}} = \mathbf{1},
$$

that is, the vector whose entries are all equal to one.  
Then the right-hand side was then constructed as

$$
b = A x_{\text{true}}.
$$

This guarantees that the linear system $Ax = b$ has a known exact solution, allowing us to evaluate the accuracy of the computed GMRES approximation by measuring the error

$$
\|x - x_{\text{true}}\|_2.
$$

For the iterative procedure, we chose the initial guess

$$
x_0 = 0,
$$

the zero vector.

#### Interpretation of Residual and Solution Differences

The residual at iteration \( j \) is defined as

$$
r_j = b - A x_j.
$$

Its norm,

$$
\|r_j\|_2,
$$

measures how well the current approximation \( x_j \) satisfies the linear system \( Ax = b \).  
If the residual norm is small, then \( x_j \) is a good approximate solution.  
In GMRES with progressive Givens rotations, this quantity is obtained efficiently as

$$
\|r_j\|_2 = |g_{j+1}|,
$$

without explicitly computing \( b - A x_j \).

Moreover, the quantity

$$
\|x_{\text{basic/givens}} - x_{\text{true}}\|_2,
$$

represents the difference between the solutions computed by: the basic GMRES implementation in Q1,or the progressive Givens implementation in Q3 with the true solution.

#### Iteration Limit Setting

In our experiments, we set the maximum Krylov subspace dimension to  
$$
m = 3000.
$$

This means that GMRES is allowed to perform at most 3000 Arnoldi iterations.  
The iteration terminates when the residual norm satisfies

$$
\|r_j\|_2 < \text{tolerance},
$$

or when the maximum dimension \( m = 3000 \) is reached or a breakdown occurs, that is,
$$
h_{j+1,j} = 0,
$$
which implies that no further Krylov basis vectors can be generated.

If the algorithm reaches 3000 iterations without the residual norm falling below the prescribed tolerance, this indicates that the method has not converged within the allowed subspace size. In such cases, the stopping criterion is triggered by the iteration limit rather than by convergence. This suggests the need for a larger subspace dimension.

In [117]:
files = [
    "bcspwr01.mtx",
    "bcspwr02.mtx",
    "gre_115.mtx",
    "494_bus.mtx",
    "1138_bus.mtx",
    "bcspwr06.mtx",
    "bcspwr10.mtx",
    "bcsstk18.mtx"
]

for fname in files:
    A= mmread(fname).tocsr()
    n= A.shape[0]
    x_true= np.ones(n)
    b= A @ x_true
    x0 = np.zeros(n)
    x_givens, residuals = gmres_givens_R(A, b, x0, m=3000, tol=1e-14)
    x_basic, res_basic = gmres(A, b, x0, m=3000)

    print(f"\n{fname}  A.shape={A.shape}  nnz={A.nnz}")
    print(f"Basic:  residual={res_basic:.3e}  error={np.linalg.norm(x_basic - x_true):.3e}")
    print(f"Givens: residual={residuals[-1]:.3e}  error={np.linalg.norm(x_givens - x_true):.3e}  iters={len(residuals)}")


bcspwr01.mtx  A.shape=(39, 39)  nnz=131
Basic:  residual=4.544e-14  error=1.364e-13
Givens: residual=1.913e-15  error=7.607e-15  iters=38

bcspwr02.mtx  A.shape=(49, 49)  nnz=167
Basic:  residual=4.678e-14  error=2.213e-01
Givens: residual=1.815e-15  error=2.500e-01  iters=42

gre_115.mtx  A.shape=(115, 115)  nnz=421
Basic:  residual=1.246e-14  error=1.606e-14
Givens: residual=4.565e-15  error=1.563e-14  iters=81

494_bus.mtx  A.shape=(494, 494)  nnz=1666
Basic:  residual=5.936e-11  error=1.270e-11
Givens: residual=9.438e-15  error=1.888e-10  iters=921

1138_bus.mtx  A.shape=(1138, 1138)  nnz=4054
Basic:  residual=1.757e-10  error=2.196e-09
Givens: residual=9.968e-15  error=1.394e-09  iters=2233

bcspwr06.mtx  A.shape=(1454, 1454)  nnz=5300
Basic:  residual=6.050e-13  error=2.694e-01
Givens: residual=9.980e-15  error=1.348e+00  iters=2644

bcspwr10.mtx  A.shape=(5300, 5300)  nnz=21842
Basic:  residual=3.032e-04  error=1.224e-01
Givens: residual=3.032e-04  error=1.224e-01  iters=3000



## Conclusion

For small systems (39×39, 49×49, and 115×115), both the basic least-squares and Givens-based implementations converge rapidly. It drives residual norms close to machine precision with correspondingly tiny solution errors. As the problem size grows (494×494, 1138×1138, and 1454×1454), GMRES still achieves small residuals, but requires much more iterations, and solution errors varie due to conditioning effects. At the largest scales (5300×5300 and 11948×11948), the algorithm hit the maximum iteration limit before the residual fell below the prescribed tolerance. So they did not truly converge. The subspace simply wasn't large enough to handle these problems.

Across most test cases, the Givens rotation variant achieved smaller final residual norms than the basic least-squares approach, which reflects the fact that Givens rotations applied incrementally to the upper Hessenberg system avoid accumulating the rounding errors that arise when solving the least-squares problem in a single batch, leading to a more accurate residual estimate at each step.

A low residual does not guarantee a small solution error when the matrix is ill-conditioned. The bcspwr02 matrix illustrates this even at small scale despite a tiny residual, the solution error remains relatively large. The bcspwr06 matrix tells the same story at larger scale. 

The experiments confirm that GMRES with Givens rotations consistently achieves smaller residual norms than the basic least-squares implementation across the sparse matrix suite, with the advantage becoming increasingly evident as problem size and iteration count grow.